# Real PIA ICAO mappings

Add the real ICAO from OpenAirframes by joining the Jacksweeney tail-number mapping on `Tail Number`.

In [1]:
from pathlib import Path
import sys
import polars as pl
from data_engineering.openairframes.read import scan_latest_faa_registration_icao_for_join

In [7]:
from pia.evaluation.real_flights_1 import df_flights
df = df_flights
from airports.airportLookup import AirportLookup
airport_lookup = AirportLookup()
df = df.with_columns(
    pl.col("takeoff_airport_ident")
    .map_elements(airport_lookup.airport_ident_is_in_us_faa_airspace, return_dtype=pl.Boolean)
    .alias("takeoff_airport_in_us_faa_airspace"),
    pl.col("landing_airport_ident")
    .map_elements(airport_lookup.airport_ident_is_in_us_faa_airspace, return_dtype=pl.Boolean)
    .alias("landing_airport_in_us_faa_airspace"),
)

In [10]:
def fix_route_mismatches(df: pl.DataFrame) -> pl.DataFrame:
    return (
        df
        .with_row_index("_route_fix_row_nr")
        .sort(["icao", "takeoff_time"])
        .with_columns(
            pl.col("takeoff_airport_ident")
            .shift(-1)
            .over("icao")
            .alias("_next_takeoff_airport_ident")
        )
        .with_columns(
            pl.when(
                pl.col("_next_takeoff_airport_ident").is_not_null()
                & (pl.col("landing_airport_ident") != pl.col("_next_takeoff_airport_ident"))
            )
            .then(pl.col("_next_takeoff_airport_ident"))
            .otherwise(pl.col("landing_airport_ident"))
            .alias("landing_airport_ident")
        )
        .sort("_route_fix_row_nr")
        .drop(["_route_fix_row_nr", "_next_takeoff_airport_ident"])
    )


df = fix_route_mismatches(df)
df_flights = fix_route_mismatches(df_flights)

route_mismatches = (
    df
    .sort(["icao", "takeoff_time"])
    .with_columns(
        pl.col("landing_airport_ident").shift(1).over("icao").alias("prev_landing_airport_ident"),
        pl.col("landing_time").shift(1).over("icao").alias("prev_landing_time"),
    )
    .filter(
        pl.col("prev_landing_airport_ident").is_not_null()
        & (pl.col("prev_landing_airport_ident") != pl.col("takeoff_airport_ident"))
    )
)

route_mismatches = route_mismatches.filter(pl.col("takeoff_airport_in_us_faa_airspace") & pl.col("landing_airport_in_us_faa_airspace"))
from airports.airportLookup import AirportLookup, haversine

airport_lookup = AirportLookup()

def airport_distance_km(row):
    prev_landing = airport_lookup.get_Airport_from_airport_ident(
        row["prev_landing_airport_ident"]
    )
    next_takeoff = airport_lookup.get_Airport_from_airport_ident(
        row["takeoff_airport_ident"]
    )

    if prev_landing is None or next_takeoff is None:
        return None

    return haversine(
        prev_landing.lat,
        prev_landing.lon,
        next_takeoff.lat,
        next_takeoff.lon,
    )

route_mismatches = route_mismatches.with_columns(
    pl.struct(["prev_landing_airport_ident", "takeoff_airport_ident"])
    .map_elements(airport_distance_km, return_dtype=pl.Float64)
    .alias("airport_disagreement_distance_km")
)
route_mismatches

icao,callsign,registration,takeoff_time,takeoff_airport_ident,takeoff_airport_ident_distance_km,landing_time,landing_airport_ident,landing_airport_ident_distance_km,last_lat,last_lon,last_baro_altitude_ft,last_geom_altitude_ft,takeoff_airport_elevation_difference_ft,landing_airport_elevation_difference_ft,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category,takeoff_airport_in_us_faa_airspace,landing_airport_in_us_faa_airspace,prev_landing_airport_ident,prev_landing_time,airport_disagreement_distance_km
str,str,str,datetime[ms],str,f64,datetime[ms],str,f64,f64,f64,i64,i64,i64,i64,bool,bool,bool,bool,str,str,str,str,bool,bool,str,datetime[ms],f64


In [11]:
df_flights

icao,callsign,registration,takeoff_time,takeoff_airport_ident,takeoff_airport_ident_distance_km,landing_time,landing_airport_ident,landing_airport_ident_distance_km,last_lat,last_lon,last_baro_altitude_ft,last_geom_altitude_ft,takeoff_airport_elevation_difference_ft,landing_airport_elevation_difference_ft,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,f64,datetime[ms],str,f64,f64,f64,i64,i64,i64,i64,bool,bool,bool,bool,str,str,str,str
"""a09218""","""FFL1923""",null,2026-05-01 16:58:04.220,"""PHNL""",null,2026-05-01 22:46:31.990,"""PHNL""",null,null,null,null,null,null,null,true,false,false,false,null,null,null,"""A3"""
"""a0ef95""","""N16CP""","""N16CP""",2026-05-01 12:25:02.100,"""KROG""",null,2026-05-01 13:18:05.620,"""KBNA""",null,null,null,null,null,null,null,false,true,false,false,"""C680""","""WALMART INC""","""CESSNA 680 Citation Sovereign""","""A2"""
"""a0ef95""","""N16CP""","""N16CP""",2026-05-01 20:34:22.270,"""KBNA""",null,2026-05-01 21:43:16.480,"""KROG""",null,null,null,null,null,null,null,false,true,false,false,"""C680""","""WALMART INC""","""CESSNA 680 Citation Sovereign""","""A2"""
"""a12e57""","""VTE3388""","""N17513""",2026-05-01 02:25:17.040,"""KPHX""",null,2026-05-01 02:44:24.170,"""AZ82""",null,null,null,null,null,null,null,false,false,false,false,"""E135""","""CORPORATE FLIGHT MANAGEMENT IN…","""EMBRAER ERJ-135""","""A2"""
"""a12e57""","""VTE3383""","""N17513""",2026-05-01 15:18:54.180,"""AZ82""",null,2026-05-01 15:41:33.750,"""KPHX""",null,null,null,null,null,null,null,false,false,false,false,"""E135""","""CORPORATE FLIGHT MANAGEMENT IN…","""EMBRAER ERJ-135""","""A2"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""a89549""","""N652MP""","""N652MP""",2026-06-08 10:40:52.880,"""KHPN""",null,2026-06-08 15:39:37.460,"""KNUQ""",null,null,null,null,null,null,null,false,true,false,false,"""GLF6""","""TVPX AIRCRAFT SOLUTIONS INC TR…","""GULFSTREAM G650""","""A3"""
"""a89549""","""N652MP""","""N652MP""",2026-06-08 17:11:26.600,"""KNUQ""",null,2026-06-08 17:33:22.430,"""KSFO""",null,null,null,null,null,null,null,false,true,false,false,"""GLF6""","""TVPX AIRCRAFT SOLUTIONS INC TR…","""GULFSTREAM G650""","""A3"""
"""a907ae""","""N681HS""","""N681HS""",2026-06-08 16:31:48.250,"""KBUR""",null,2026-06-08 19:00:50.320,"""KAFW""",null,null,null,null,null,null,null,false,true,false,false,"""C680""","""TVPX AIRCRAFT SOLUTIONS INC TR…","""CESSNA 680 Citation Sovereign""","""A2"""


# see when real icaos last appeared